In [ ]:
!pip install lexical_diversity
!pip install textstat
!pip install sacrebleu
!pip install lexicalrichness
!pip install mauve-text

In [2]:
import pandas as pd
from lexical_diversity import lex_div as ld
import textstat
import numpy as np
import sacrebleu
from lexicalrichness import LexicalRichness
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import pdist, squareform
import mauve
from nltk import ngrams

In [4]:
df_all_aug = pd.read_csv('/content/All_aug - full.tsv', sep='\t')[:2204].iloc[:, 2].astype(str).tolist()
df_gpt = pd.read_csv('/content/GPT_errorifier - full.tsv', sep='\t')[:2204].iloc[:, 2].astype(str).tolist()
df_llm = pd.read_csv('/content/LLM_errorifier - full.tsv', sep='\t')[:2204].iloc[:, 2].astype(str).tolist()
df_rule = pd.read_csv('/content/Rule_based_errorifier - full.tsv', sep='\t')[:2204].iloc[:, 2].astype(str).tolist()
df_real = pd.read_csv('real_errors.tsv.txt', sep='\t')[:2204].iloc[:, 2].astype(str).tolist()

/tmp/ipykernel_1326/2988708787.py:4: DtypeWarning: Columns (360) have mixed types. Specify dtype option on import or set low_memory=False.
  df_rule = pd.read_csv('/content/Rule_based_errorifier - full.tsv', sep='\t')[:2204].iloc[:, 2].astype(str).tolist()


In [5]:
def calculate_lexical_diversity(sentences):
    """Calculates the Measure of Textual Lexical Diversity (MTLD)."""
    combined_text = " ".join(sentences)
    lex = LexicalRichness(combined_text)
    return lex.mtld()

In [6]:
print(calculate_lexical_diversity(df_all_aug),
      calculate_lexical_diversity(df_gpt),
      calculate_lexical_diversity(df_llm),
      calculate_lexical_diversity(df_rule),
      calculate_lexical_diversity(df_real))

77.06776614342783 62.41649296912591 101.79152374903236 89.34448495720295 56.05232836324829


In [7]:
print(textstat.flesch_reading_ease(' '.join(df_all_aug)),
      textstat.flesch_reading_ease(' '.join(df_gpt)),
      textstat.flesch_reading_ease(' '.join(df_llm)),
      textstat.flesch_reading_ease(' '.join(df_rule)),
      textstat.flesch_reading_ease(' '.join(df_real))
      )

63.63041581220595 65.49909887639848 57.68569824553282 61.06083280163462 58.3056414512793


In [8]:
def calculate_self_bleu(sentences, sample_size=500):
    """Calculates Self-BLEU to quantify internal redundancy."""
    if len(sentences) > sample_size:
        indices = np.random.choice(len(sentences), sample_size, replace=False)
        subset = [sentences[i] for i in indices]
    else:
        subset = sentences

    scores = []
    for i in range(len(subset)):
        hypothesis = subset[i]
        references = [subset[:i] + subset[i+1:]]
        bleu = sacrebleu.sentence_bleu(hypothesis, references[0]).score
        scores.append(bleu)
    return np.mean(scores)

In [21]:
print(calculate_self_bleu(df_all_aug),
      calculate_self_bleu(df_gpt),
      calculate_self_bleu(df_llm),
      calculate_self_bleu(df_rule),
      calculate_self_bleu(df_real))

23.159164015641124 25.683552627356416 29.013212096901835 26.962022239769603 29.970949315812387


In [9]:
out = mauve.compute_mauve(p_text=df_real, q_text=df_all_aug, device_id=0, verbose=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.25G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/436 [00:00<?, ?it/s]

GPT2Model LOAD REPORT from: gpt2-large
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...35}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Featurizing p:   0%|          | 0/2204 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/2204 [00:00<?, ?it/s]

In [10]:
out.mauve

np.float64(0.4796448186115153)

In [11]:
out_gpt = mauve.compute_mauve(p_text=df_real, q_text=df_gpt, device_id=0, verbose=False)

Featurizing p:   0%|          | 0/2204 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/2204 [00:00<?, ?it/s]

In [12]:
out_llm = mauve.compute_mauve(p_text=df_real, q_text=df_llm, device_id=0, verbose=False)

Featurizing p:   0%|          | 0/2204 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/2204 [00:00<?, ?it/s]

In [16]:
out_rule = mauve.compute_mauve(p_text=df_real, q_text=df_rule, device_id=0, verbose=False)

Featurizing p:   0%|          | 0/2204 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/2204 [00:00<?, ?it/s]

In [17]:
print(out.mauve,
      out_gpt.mauve,
      out_llm.mauve,
      out_rule.mauve)

0.4796448186115153 0.47405007685672207 0.9654126319475689 0.3647684028358592


In [18]:
def get_ngrams(corpus, n):
    all_ngrams = []
    for sent in corpus:
        tokens = sent.lower().split()
        all_ngrams.extend(list(ngrams(tokens, n)))
    return set(all_ngrams)

In [19]:
def calculate_ngram_metrics(real_sentences, synth_sentences, n=2):
    """Calculates n-gram overlap and novelty between two corpora."""
    real_ngrams = get_ngrams(real_sentences, n)
    synth_ngrams = get_ngrams(synth_sentences, n)
    intersection = real_ngrams.intersection(synth_ngrams)
    union = real_ngrams.union(synth_ngrams)
    jaccard = len(intersection) / len(union) if len(union) > 0 else 0
    novelty = 1 - (len(intersection) / len(synth_ngrams)) if len(synth_ngrams) > 0 else 0
    return jaccard, novelty

In [20]:
print(calculate_ngram_metrics(df_real, df_all_aug),
      calculate_ngram_metrics(df_real, df_gpt),
      calculate_ngram_metrics(df_real, df_llm),
      calculate_ngram_metrics(df_real, df_rule))

(0.1362991008045433, 0.7561388653683319) (0.279416902213612, 0.5498592990534663) (0.6583964800782205, 0.1906145139480142) (0.12736559298514954, 0.7527085124677558)


In [23]:
def load_data(path, limit=2204):
    df = pd.read_csv(path, sep='\t')
    return df.iloc[:limit, 2].astype(str).tolist()

In [24]:
datasets = {
    "All_Augmented": load_data('/content/All_aug - full.tsv'),
    "GPT_Errorifier": load_data('/content/GPT_errorifier - full.tsv'),
    "LLM_Errorifier": load_data('/content/LLM_errorifier - full.tsv'),
    "Rule_Based": load_data('/content/Rule_based_errorifier - full.tsv'),
    "Real_Learner": load_data('real_errors.tsv.txt')
}

/tmp/ipykernel_1326/585785809.py:2: DtypeWarning: Columns (360) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep='\t')


In [25]:
def get_ngram_stats(synth, real, n=3):
    def get_set(corpus):
        res = set()
        for s in corpus:
            tokens = s.lower().split()
            res.update(ngrams(tokens, n))
        return res

    s_set, r_set = get_set(synth), get_set(real)
    intersection = s_set.intersection(r_set)
    jaccard = len(intersection) / len(s_set.union(r_set)) if s_set else 0
    novelty = 1 - (len(intersection) / len(s_set)) if s_set else 0
    return jaccard, novelty

In [27]:
results = []
real_data = datasets["Real_Learner"]

for name, data in datasets.items():
    print(f"Processing {name}...")

    # Internal Diversity Metrics
    mtld_val = calculate_lexical_diversity(data)
    flesch_val = textstat.flesch_reading_ease(" ".join(data[:100])) # Proxied on subset for speed
    self_bleu = calculate_self_bleu(data, sample_size=100)

    # Comparative Metrics (vs Real Learner)
    if name != "Real_Learner":
        m_score = mauve.compute_mauve(p_text=real_data, q_text=data, device_id=0, verbose=False).mauve
        jaccard, novelty = get_ngram_stats(data, real_data, n=3)
    else:
        m_score, jaccard, novelty = 1.0, 1.0, 0.0

    results.append({
        "Pipeline": name,
        "MTLD (Lexical)": round(mtld_val, 2),
        "Flesch Ease": round(flesch_val, 2),
        "Self-BLEU ↓": round(self_bleu, 2),
        "MAUVE Score ↑": round(m_score, 4),
        "3-gram Overlap": round(jaccard, 4),
        "3-gram Novelty": round(novelty, 4)
    })

Processing All_Augmented...


Featurizing p:   0%|          | 0/2204 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/2204 [00:00<?, ?it/s]

Processing GPT_Errorifier...


Featurizing p:   0%|          | 0/2204 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/2204 [00:00<?, ?it/s]

... (3 outputs left out)

Processing Rule_Based...


Featurizing p:   0%|          | 0/2204 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/2204 [00:00<?, ?it/s]

Processing Real_Learner...


In [28]:
evaluation_table = pd.DataFrame(results)
display(evaluation_table)
print(evaluation_table.to_latex(index=False))

,Pipeline,MTLD (Lexical),Flesch Ease,Self-BLEU ↓,MAUVE Score ↑,3-gram Overlap,3-gram Novelty
0,All_Augmented,77.07,63.23,14.03,0.4796,0.0462,0.9163
1,GPT_Errorifier,62.42,64.23,15.25,0.4741,0.1850,0.7024
2,LLM_Errorifier,101.79,58.29,12.24,0.9654,0.6360,0.1965
3,Rule_Based,89.34,64.83,12.75,0.3648,0.0411,0.9185
4,Real_Learner,56.05,59.24,16.37,1.0000,1.0000,0.0000


\begin{tabular}{lrrrrrr}
\toprule
Pipeline & MTLD (Lexical) & Flesch Ease & Self-BLEU ↓ & MAUVE Score ↑ & 3-gram Overlap & 3-gram Novelty \\
\midrule
All_Augmented & 77.070000 & 63.230000 & 14.030000 & 0.479600 & 0.046200 & 0.916300 \\
GPT_Errorifier & 62.420000 & 64.230000 & 15.250000 & 0.474100 & 0.185000 & 0.702400 \\
LLM_Errorifier & 101.790000 & 58.290000 & 12.240000 & 0.965400 & 0.636000 & 0.196500 \\
Rule_Based & 89.340000 & 64.830000 & 12.750000 & 0.364800 & 0.041100 & 0.918500 \\
Real_Learner & 56.050000 & 59.240000 & 16.370000 & 1.000000 & 1.000000 & 0.000000 \\
\bottomrule
\end{tabular}

